# Data_Loading_and_Variable_Assigning — part 1 of 1

This notebook was automatically split from a larger notebook.

In [3]:
import importlib

# List of libraries to check
libraries = [
    'numpy',
    'matplotlib',
    'scipy',
    'torch',
    'torchdiffeq',
    'torchsummary',
    'minepy',
    'pyrqa',
    'pyts',
    'MFDFA',
    'pyinform',
    'graphviz',
    'fa2',
    'networkx'
]

for lib in libraries:
    try:
        module = importlib.import_module(lib)
        version = getattr(module, '__version__', 'Unknown version')
        print(f'{lib}: {version}')
    except ImportError:
        print(f'{lib} is not installed.')

numpy: 2.4.2
matplotlib: 3.10.8
scipy: 1.17.0
torch is not installed.
torchdiffeq is not installed.
torchsummary is not installed.
minepy is not installed.
pyrqa is not installed.
pyts is not installed.
MFDFA is not installed.
pyinform is not installed.
graphviz is not installed.
fa2 is not installed.
networkx is not installed.


In [4]:
import mat73

data = mat73.loadmat('/home/a/projects/Complete-Neural-Signal-Analysis/EEG_DS_Struct_0101.mat')
print(data.keys())
print(type(data['DSamp']))
print(data['DSamp'])

dict_keys(['DSamp'])
<class 'dict'>
{'EEGdata': array([[-21295.98864928, -21303.74707676, -21315.46657134, ...,
        -17286.89230419, -17281.27799398, -17330.52269117],
       [-20109.7167273 , -20120.74615359, -20130.12657698, ...,
        -16618.13723294, -16602.343699  , -16641.89672243],
       [-24153.38375243, -24163.86401194, -24171.94434272, ...,
        -14755.59815419, -14743.08142526, -14783.66201823],
       ...,
       [  2692.44573979,   2703.97118199,   2731.75987441, ...,
          4181.64344633,   4157.70728788,   4198.03945844],
       [ -5014.9543463 ,  -5014.10209995,  -5019.83464311, ...,
         -4711.95486888,  -4706.99851768,  -4709.83053726],
       [    76.88986022,     72.02278487,     66.20816171, ...,
            73.4410151 ,     69.33401454,     66.65666327]]), 'Subj': '0101', 'fs': array(1000.), 'fsOld': array(2000.), 'label': [['Fp1'], ['Fpz'], ['Fp2'], ['F7'], ['F3'], ['Fz'], ['F4'], ['F8'], ['FC5'], ['FC1'], ['FC2'], ['FC6'], ['M1'], ['T7'], ['C3']

In [3]:
import numpy as np
import pandas as pd
import mat73

# -------------------------------------------------------
# LOAD DATA
# -------------------------------------------------------
mat_path = '/home/a/projects/Complete-Neural-Signal-Analysis/EEG_DS_Struct_0101.mat'
stim_path = '/home/a/projects/Complete-Neural-Signal-Analysis/stim_data.xlsx'

raw_data = mat73.loadmat(mat_path)
stim_data = pd.read_excel(stim_path)

# Fill null values in 'Sub#' column
stim_data['Sub#'] = stim_data['Sub#'].ffill()

# -------------------------------------------------------
# HELPERS TO REBUILD LEGACY-STYLE DSamp
# -------------------------------------------------------
def scalar_to_2d(x):
    arr = np.asarray(x).squeeze()
    if arr.size == 0:
        return np.array([[np.nan]])
    if arr.size == 1:
        v = arr.item()
        if isinstance(v, (float, np.floating)) and float(v).is_integer():
            v = int(v)
        return np.array([[v]])
    return np.array([[arr]], dtype=object)

def vector_to_1wrap(x):
    if x is None:
        return np.array([np.array([])], dtype=object)
    return np.array([np.asarray(x).squeeze()], dtype=object)

def subject_to_1wrap(x):
    if x is None:
        return np.array([''], dtype=object)
    return np.array([str(x)], dtype=object)

def colvec(x):
    arr = np.asarray(x).squeeze()
    return arr.reshape(-1, 1)

def flatten(item):
    if isinstance(item, list):
        for subitem in item:
            yield from flatten(subitem)
    elif isinstance(item, tuple):
        for subitem in item:
            yield from flatten(subitem)
    elif isinstance(item, np.ndarray):
        for subitem in item.flatten():
            yield from flatten(subitem)
    else:
        yield item

def force_list(x, default=None):
    if x is None:
        return [default] if default is not None else []
    vals = list(flatten(x))
    if len(vals) == 0 and default is not None:
        return [default]
    return vals

def unwrap_scalar(x, default=np.nan):
    if x is None:
        return default
    arr = np.asarray(x)
    if arr.size == 0:
        return default
    val = arr.flatten()[0]
    try:
        if isinstance(val, (float, np.floating)) and np.isnan(val):
            return default
    except Exception:
        pass
    return val

def unwrap_string(x, default='Unknown'):
    if x is None:
        return default
    arr = np.asarray(x, dtype=object)
    if arr.size == 0:
        return default
    val = arr.flatten()[0]
    if val is None:
        return default
    sval = str(val).strip()
    if sval == '':
        return default
    return sval

def simplify_labels(label_obj):
    out = []
    for item in label_obj:
        vals = force_list(item, default='Unknown')
        out.append(str(vals[0]))
    return out

def labels_to_legacy(label_obj):
    # target style:
    # [[array(['Fp1'])],
    #  [array(['Fpz'])], ...]
    simple = simplify_labels(label_obj)
    out = np.empty((len(simple), 1), dtype=object)
    for i, lbl in enumerate(simple):
        out[i, 0] = np.array([lbl])
    return out

def stim_to_legacy_array(x):
    arr = np.asarray(x, dtype=object)
    if arr.size == 0:
        return np.empty((1, 0), dtype=float)
    val = arr.flatten()[0]
    if val is None:
        return np.empty((1, 0), dtype=float)
    sval = str(val).strip()
    if sval == '' or sval.lower() in ('unknown', 'none', 'nan'):
        return np.empty((1, 0), dtype=float)
    return np.array([sval], dtype=object)

def triggers_to_legacy(triggers_obj):
    """
    Rebuild triggers to match the legacy scipy loadmat-like structure:
    [[
      (array([[20.428]]), array([[20429]]), array(['0002']), array([[2]]),
       array(['Block Start']), array([], shape=(1,0)))
      ...
    ]]
    """
    if triggers_obj is None:
        return np.empty((1, 0), dtype=object)

    triggers_iter = list(triggers_obj)
    normalized = []

    for tr in triggers_iter:
        if isinstance(tr, dict):
            # These are your actual keys from mat73:
            # {'Label': ..., 'StimType': ..., 'code': ..., 'offset': ..., 'time': ..., 'type': ...}
            t = unwrap_scalar(tr.get('time', None), np.nan)
            sample_num = unwrap_scalar(tr.get('offset', None), np.nan)
            file_num = unwrap_string(tr.get('code', None), 'Unknown')
            event_type = unwrap_scalar(tr.get('type', None), np.nan)
            event_desc = unwrap_string(tr.get('Label', None), 'Unknown')
            stim_type = tr.get('StimType', None)

            tup = (
                np.array([[float(t)]]) if not (isinstance(t, float) and np.isnan(t)) else np.array([[np.nan]]),
                np.array([[int(sample_num)]]) if not (isinstance(sample_num, float) and np.isnan(sample_num)) else np.array([[np.nan]]),
                np.array([file_num]),
                np.array([[int(event_type)]]) if not (isinstance(event_type, float) and np.isnan(event_type)) else np.array([[np.nan]]),
                np.array([event_desc]),
                stim_to_legacy_array(stim_type)
            )
            normalized.append(tup)
        else:
            tr_list = list(tr)
            while len(tr_list) < 6:
                tr_list.append(np.empty((1, 0), dtype=float))
            normalized.append(tuple(tr_list[:6]))

    out = np.empty((1, len(normalized)), dtype=object)
    for i, item in enumerate(normalized):
        out[0, i] = item
    return out

# -------------------------------------------------------
# REBUILD DSamp SO OLD INDEXING STILL WORKS
# -------------------------------------------------------
DSamp_raw = raw_data['DSamp']

triggers_raw = DSamp_raw.get('triggers', None)
EEGdata_raw = np.asarray(DSamp_raw.get('EEGdata'))
fs_raw = DSamp_raw.get('fs', np.nan)
fsOld_raw = DSamp_raw.get('fsOld', np.nan)
time_raw = DSamp_raw.get('time', np.array([]))
label_raw = DSamp_raw.get('label', [])
nchan_raw = DSamp_raw.get('nchan', np.nan)
rate_raw = DSamp_raw.get('rate', np.nan)
npt_raw = DSamp_raw.get('npt', np.nan)
Subj_raw = DSamp_raw.get('Subj', '')
ptrackerPerf_raw = DSamp_raw.get('ptrackerPerf', np.array([]))
ptrackerTime_raw = DSamp_raw.get('ptrackerTime', np.array([]))
ptrackerfs_raw = DSamp_raw.get('ptrackerfs', np.nan)

triggers_legacy = triggers_to_legacy(triggers_raw)
EEGdata_legacy = EEGdata_raw
fs_legacy = scalar_to_2d(fs_raw)
fsOld_legacy = scalar_to_2d(fsOld_raw)
time_legacy = vector_to_1wrap(time_raw)
label_legacy = labels_to_legacy(label_raw)
nchan_legacy = scalar_to_2d(nchan_raw)
rate_legacy = scalar_to_2d(rate_raw)
npt_legacy = scalar_to_2d(npt_raw)
Subj_legacy = subject_to_1wrap(Subj_raw)
ptrackerPerf_legacy = colvec(ptrackerPerf_raw)
ptrackerTime_legacy = colvec(ptrackerTime_raw)
ptrackerfs_legacy = scalar_to_2d(ptrackerfs_raw)

DSamp = np.empty((1, 1), dtype=object)
DSamp[0, 0] = [
    triggers_legacy,
    EEGdata_legacy,
    fs_legacy,
    fsOld_legacy,
    time_legacy,
    label_legacy,
    nchan_legacy,
    rate_legacy,
    npt_legacy,
    Subj_legacy,
    ptrackerPerf_legacy,
    ptrackerTime_legacy,
    ptrackerfs_legacy
]

data = {'DSamp': DSamp}

# -------------------------------------------------------
# ORIGINAL OUTPUT LOGIC STAYS THE SAME
# -------------------------------------------------------
DSamp = data['DSamp']

# Get data parameters
triggers = DSamp[0][0][0]
EEGdata = DSamp[0][0][1]
fs = DSamp[0][0][2][0][0]
fsOld = DSamp[0][0][3][0][0]
time = DSamp[0][0][4][0]
label = DSamp[0][0][5]
nchan = DSamp[0][0][6][0][0]
rate = DSamp[0][0][7][0][0]
npt = DSamp[0][0][8][0][0]
Subj = DSamp[0][0][9][0]
ptrackerPerf = DSamp[0][0][10]
ptrackerTime = DSamp[0][0][11]
ptrackerfs = DSamp[0][0][12][0][0]

# List of unwanted channel names
unwanted_channels = ['BIP1', 'BIP2', 'RESP1']

# Create a mask where True indicates that the channel is not unwanted
mask = np.array([ch[0][0] not in unwanted_channels for ch in label])

# Filter out unwanted channels from the label data
filtered_label = label[mask]

# Convert the filtered list back to numpy array and replace the original label
label = np.array(filtered_label, dtype=object)

# Transpose EEGdata
EEGdata = EEGdata.T

# Filter out unwanted channels from the EEG data
filtered_EEGdata = EEGdata[:, mask]

# Transpose it back if needed
filtered_EEGdata = filtered_EEGdata.T

# Select subject
stim_data = stim_data[stim_data['Sub#'] == 1]
stim_data_df = pd.DataFrame(stim_data)

trigger_list = []
for trigger in triggers[0]:
    trigger_list.append([
        list(flatten(trigger[0])),  # Time
        list(flatten(trigger[1])),  # SampleNum
        list(flatten(trigger[2])),  # FileNum
        list(flatten(trigger[3])),  # EventType
        list(flatten(trigger[4])),  # EventDescription
        list(flatten(trigger[5])) if np.size(trigger[5]) else ['Unknown']  # StimType
    ])

# Create DataFrame and transpose it
eeg_df = pd.DataFrame(filtered_EEGdata.T)

# Convert labels into a simple list
simple_label = [label_item[0][0] for label_item in label]
eeg_df.columns = simple_label

# Create triggers DataFrame
triggers_df = pd.DataFrame(
    trigger_list,
    columns=["Time", "SampleNum", "EventType1", "EventType", "EventDescription", "StimType"]
)
triggers_df = triggers_df.drop(['EventType1', 'SampleNum'], axis=1)

# -------------------------------------------------------
# OPTIONAL CHECKS
# -------------------------------------------------------
print("Subject:", Subj)
print("fs:", fs)
print("fsOld:", fsOld)
print("nchan:", nchan)
print("npt:", npt)
print("EEG raw shape:", np.asarray(DSamp[0][0][1]).shape)
print("EEG filtered DataFrame shape:", eeg_df.shape)
print("Stim DataFrame shape:", stim_data_df.shape)
print("Triggers DataFrame shape:", triggers_df.shape)
print(triggers_df.head())

Subject: 0101
fs: 1000
fsOld: 2000
nchan: 35
npt: 4227788
EEG raw shape: (35, 4227788)
EEG filtered DataFrame shape: (4227788, 32)
Stim DataFrame shape: (6, 18)
Triggers DataFrame shape: (24, 4)
        Time EventType EventDescription   StimType
0   [20.428]       [2]    [Block Start]  [Unknown]
1  [619.442]       [2]    [Block Start]  [Unknown]
2  [619.499]      [16]     [Stim Start]      [M30]
3  [654.746]      [32]      [Stim Stop]  [Unknown]
4  [770.515]      [16]     [Stim Start]      [M30]


In [4]:
print(type(DSamp_raw['triggers']))
print(type(DSamp_raw['triggers'][0]))
print(DSamp_raw['triggers'][0])

<class 'list'>
<class 'dict'>
{'Label': 'Block Start', 'StimType': None, 'code': '0002', 'offset': array(20429.), 'time': array(20.428), 'type': array(2.)}


In [5]:
import numpy as np
import pandas as pd

# -------------------------------------------------------
# NORMALIZE REAL triggers_df
# -------------------------------------------------------
def unwrap_singleton(x):
    while isinstance(x, (list, tuple, np.ndarray)) and len(x) == 1:
        x = x[0]
    return x

for col in ["Time", "EventDescription", "StimType"]:
    if col in triggers_df.columns:
        triggers_df[col] = triggers_df[col].apply(unwrap_singleton)

triggers_df["Time"] = pd.to_numeric(triggers_df["Time"], errors="coerce")
triggers_df["EventDescription"] = triggers_df["EventDescription"].astype(str).str.strip()
triggers_df["StimType"] = triggers_df["StimType"].astype(str).str.strip()

# Keep only rows with valid time
triggers_df = triggers_df[triggers_df["Time"].notna()].copy()

# -------------------------------------------------------
# SUBJECT / SESSION TAGS
# -------------------------------------------------------
# Adjust these if needed
sub_num = 1
session_num = 1

triggers_df["Sub#"] = sub_num
triggers_df["Session"] = session_num

# Also make sure stim_data_df is filtered to the same subject/session if needed
stim_data_df = stim_data_df.copy()
stim_data_df["Sub#"] = pd.to_numeric(stim_data_df["Sub#"], errors="coerce")
if "Session" in stim_data_df.columns:
    stim_data_df["Session"] = pd.to_numeric(stim_data_df["Session"], errors="coerce")

# -------------------------------------------------------
# LEGACY MERGE LOGIC
# -------------------------------------------------------
def get_stim_info(sub, session, stim_type):
    mask = (stim_data_df["Sub#"] == sub) & (stim_data_df["Session"] == session)
    matching_rows = stim_data_df[mask]

    amplitudes = []
    blocks = []
    file_nums = []

    for _, row in matching_rows.iterrows():
        for block in range(1, 4):
            stim_col = f"StimTypeBlock{block}"
            amp_col = f"StimAmplitude_mA_block{block}"
            if stim_col in row.index and amp_col in row.index:
                if row[stim_col] == stim_type:
                    amplitudes.append(row[amp_col])
                    blocks.append(block)
                    file_nums.append(row["File Num"])

    return amplitudes, blocks, file_nums

unique_combinations = pd.concat([
    stim_data_df[["Sub#", "Session", "StimTypeBlock1"]].rename(columns={"StimTypeBlock1": "StimType"}),
    stim_data_df[["Sub#", "Session", "StimTypeBlock2"]].rename(columns={"StimTypeBlock2": "StimType"}),
    stim_data_df[["Sub#", "Session", "StimTypeBlock3"]].rename(columns={"StimTypeBlock3": "StimType"}),
    triggers_df[["Sub#", "Session", "StimType"]]
]).drop_duplicates()

results_df = pd.DataFrame(columns=["Sub#", "Session", "StimType", "Amplitudes", "Block", "File Num"])

for _, row in unique_combinations.iterrows():
    sub = row["Sub#"]
    session = row["Session"]
    stim_type = row["StimType"]

    amplitudes, blocks, file_nums = get_stim_info(sub, session, stim_type)

    for amp, block, file_num in zip(amplitudes, blocks, file_nums):
        df_temp = pd.DataFrame([{
            "Sub#": sub,
            "Session": session,
            "StimType": stim_type,
            "Amplitudes": amp,
            "Block": block,
            "File Num": file_num
        }])
        results_df = pd.concat([results_df, df_temp], ignore_index=True)

def get_stim_amplitude(sub, session, stim_type):
    mask = (
        (results_df["Sub#"] == sub) &
        (results_df["Session"] == session) &
        (results_df["StimType"] == stim_type)
    )
    matching_rows = results_df[mask]
    if not matching_rows.empty:
        return matching_rows["Amplitudes"].values[0]
    return 0.0

triggers_df["Amplitude"] = triggers_df.apply(
    lambda row: get_stim_amplitude(row["Sub#"], row["Session"], row["StimType"]),
    axis=1
)

results_df_no_amp = results_df.drop("Amplitudes", axis=1)

merged_stim_df = pd.merge(
    results_df_no_amp,
    triggers_df,
    on=["Sub#", "Session", "StimType"],
    how="inner"
)

# Convert trigger time from seconds to ms
merged_stim_df["Time"] = merged_stim_df["Time"] * 1000.0

# -------------------------------------------------------
# EEG TIME COLUMN
# -------------------------------------------------------
sampling_rate = 1000
num_samples = len(eeg_df)

# Make Time match sample index in milliseconds at 1000 Hz
eeg_df = eeg_df.copy()
eeg_df["Time"] = np.arange(num_samples, dtype=float)

# -------------------------------------------------------
# BUILD FINAL LEGACY-STYLE OUTPUT TABLE
# -------------------------------------------------------
final_eeg_df = eeg_df.copy()
final_eeg_df["Sub#"] = sub_num
final_eeg_df["Session"] = session_num
final_eeg_df["Amplitude"] = 0.0
final_eeg_df["Frequency"] = 0.0
final_eeg_df["Location"] = 0
final_eeg_df["Stim"] = 0
final_eeg_df["block"] = 0

def parse_stim_type(stim_type):
    """
    Example:
      M30 -> ('M', 30)
      F5  -> ('F', 5)
      P0  -> ('P', 0)
    """
    stim_type = str(stim_type).strip()
    letters = "".join(ch for ch in stim_type if ch.isalpha())
    digits = "".join(ch for ch in stim_type if ch.isdigit())
    freq = float(digits) if digits else 0.0

    # Numeric encoding for location so the final table stays numeric
    # Change this map if your old pipeline used a different encoding
    location_map = {"M": 1, "F": 2, "P": 3}
    location_val = location_map.get(letters, 0)

    return location_val, freq

# -------------------------------------------------------
# ANNOTATE EEG BY START/STOP INTERVALS
# -------------------------------------------------------
starts = merged_stim_df[
    merged_stim_df["EventDescription"].astype(str).str.contains("Start", case=False, na=False)
].copy()

stops = merged_stim_df[
    merged_stim_df["EventDescription"].astype(str).str.contains("Stop", case=False, na=False)
].copy()

starts = starts.sort_values(["StimType", "Time"]).reset_index(drop=True)
stops = stops.sort_values(["StimType", "Time"]).reset_index(drop=True)

for stim_type in sorted(set(starts["StimType"]).intersection(set(stops["StimType"]))):
    s_rows = starts[starts["StimType"] == stim_type].sort_values("Time").reset_index(drop=True)
    e_rows = stops[stops["StimType"] == stim_type].sort_values("Time").reset_index(drop=True)

    n_pairs = min(len(s_rows), len(e_rows))
    if n_pairs == 0:
        continue

    # Use the first matching metadata row for this stim type/session
    meta_rows = results_df[
        (results_df["Sub#"] == sub_num) &
        (results_df["Session"] == session_num) &
        (results_df["StimType"] == stim_type)
    ].reset_index(drop=True)

    for i in range(n_pairs):
        t0 = int(round(s_rows.loc[i, "Time"]))
        t1 = int(round(e_rows.loc[i, "Time"]))
        if t1 < t0:
            continue

        if len(meta_rows) > 0:
            meta_idx = min(i, len(meta_rows) - 1)
            amp = float(meta_rows.loc[meta_idx, "Amplitudes"])
            block = int(meta_rows.loc[meta_idx, "Block"])
        else:
            amp = float(s_rows.loc[i, "Amplitude"]) if "Amplitude" in s_rows.columns else 0.0
            block = 0

        location_val, freq_val = parse_stim_type(stim_type)

        final_eeg_df.loc[t0:t1, "Amplitude"] = amp
        final_eeg_df.loc[t0:t1, "Frequency"] = freq_val
        final_eeg_df.loc[t0:t1, "Location"] = location_val
        final_eeg_df.loc[t0:t1, "Stim"] = 1
        final_eeg_df.loc[t0:t1, "block"] = block

# -------------------------------------------------------
# LEGACY LABEL EXTRACTION
# -------------------------------------------------------
data_as_list = [arr.tolist()[0] for arr in DSamp[0][0][5]]

eeg_label_df = pd.DataFrame(data_as_list, columns=["EEG Electrode Labels"])

unwanted_channels = ["BIP1", "BIP2", "RESP1"]
eeg_label_df = eeg_label_df[~eeg_label_df["EEG Electrode Labels"].isin(unwanted_channels)]

# -------------------------------------------------------
# CHECKS
# -------------------------------------------------------
print("results_df shape:", results_df.shape)
print("merged_stim_df shape:", merged_stim_df.shape)
print("final_eeg_df shape:", final_eeg_df.shape)
print("eeg_label_df shape:", eeg_label_df.shape)

print("\nfinal_eeg_df head:")
print(final_eeg_df.head())

print("\nmerged_stim_df head:")
print(merged_stim_df.head())

/tmp/ipykernel_26226/2747501337.py:87: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results_df = pd.concat([results_df, df_temp], ignore_index=True)


results_df shape: (18, 6)
merged_stim_df shape: (8, 9)
final_eeg_df shape: (4227788, 40)
eeg_label_df shape: (32, 1)

final_eeg_df head:
            Fp1           Fpz           Fp2           F7         F3  \
0 -21295.988649 -20109.716727 -24153.383752  3189.340060 -45.189275   
1 -21303.747077 -20120.746154 -24163.864012  3178.880909 -56.702035   
2 -21315.466571 -20130.126577 -24171.944343  3164.903807 -69.465350   
3 -21317.809594 -20131.044726 -24174.790986  3159.478572 -73.214591   
4 -21325.798142 -20137.522181 -24179.985166  3144.934679 -84.871628   

            Fz          F4           F8          FC5          FC1  ...  \
0 -8525.066680 -642.128590  3487.913621  6324.956639  6503.012177  ...   
1 -8532.499649 -651.966372  3477.011771  6315.078704  6496.522520  ...   
2 -8544.315275 -663.772856  3463.194795  6302.391524  6483.178723  ...   
3 -8545.873916 -666.109249  3457.870782  6297.212341  6481.970244  ...   
4 -8551.164448 -671.761501  3450.466406  6283.925509  6477.045614 

In [6]:
import numpy as np
import pandas as pd

# -------------------------------------------------------
# ASSUMES THESE ALREADY EXIST FROM THE REBUILD BLOCK:
#   DSamp
#   eeg_df
#   stim_data_df
#   triggers_df
# -------------------------------------------------------

# -------------------------------------------------------
# NORMALIZE REAL triggers_df
# -------------------------------------------------------
def unwrap_singleton(x):
    while isinstance(x, (list, tuple, np.ndarray)) and len(x) == 1:
        x = x[0]
    return x

for col in ["Time", "EventDescription", "StimType"]:
    if col in triggers_df.columns:
        triggers_df[col] = triggers_df[col].apply(unwrap_singleton)

triggers_df["Time"] = pd.to_numeric(triggers_df["Time"], errors="coerce")
triggers_df["EventDescription"] = triggers_df["EventDescription"].astype(str).str.strip()
triggers_df["StimType"] = triggers_df["StimType"].astype(str).str.strip()

# Keep only valid stimulation events
triggers_df = triggers_df[triggers_df["Time"].notna()].copy()
triggers_df = triggers_df[
    triggers_df["EventDescription"].isin(["Stim Start", "Stim Stop"])
].copy()
triggers_df = triggers_df[triggers_df["StimType"] != "Unknown"].copy()

# -------------------------------------------------------
# SUBJECT / SESSION TAGS
# -------------------------------------------------------
sub_num = 1
session_num = 1

triggers_df["Sub#"] = sub_num
triggers_df["Session"] = session_num

# Make sure stim_data_df types are usable
stim_data_df = stim_data_df.copy()
stim_data_df["Sub#"] = pd.to_numeric(stim_data_df["Sub#"], errors="coerce")
if "Session" in stim_data_df.columns:
    stim_data_df["Session"] = pd.to_numeric(stim_data_df["Session"], errors="coerce")

# -------------------------------------------------------
# LEGACY MERGE LOGIC
# -------------------------------------------------------
def get_stim_info(sub, session, stim_type):
    mask = (stim_data_df["Sub#"] == sub) & (stim_data_df["Session"] == session)
    matching_rows = stim_data_df[mask]

    amplitudes = []
    blocks = []
    file_nums = []

    for _, row in matching_rows.iterrows():
        for block in range(1, 4):
            stim_col = f"StimTypeBlock{block}"
            amp_col = f"StimAmplitude_mA_block{block}"
            if stim_col in row.index and amp_col in row.index:
                if row[stim_col] == stim_type:
                    amplitudes.append(row[amp_col])
                    blocks.append(block)
                    file_nums.append(row["File Num"])

    return amplitudes, blocks, file_nums

unique_combinations = pd.concat([
    stim_data_df[["Sub#", "Session", "StimTypeBlock1"]].rename(columns={"StimTypeBlock1": "StimType"}),
    stim_data_df[["Sub#", "Session", "StimTypeBlock2"]].rename(columns={"StimTypeBlock2": "StimType"}),
    stim_data_df[["Sub#", "Session", "StimTypeBlock3"]].rename(columns={"StimTypeBlock3": "StimType"}),
    triggers_df[["Sub#", "Session", "StimType"]]
]).drop_duplicates()

results_df = pd.DataFrame(columns=["Sub#", "Session", "StimType", "Amplitudes", "Block", "File Num"])

for _, row in unique_combinations.iterrows():
    sub = row["Sub#"]
    session = row["Session"]
    stim_type = row["StimType"]

    amplitudes, blocks, file_nums = get_stim_info(sub, session, stim_type)

    for amp, block, file_num in zip(amplitudes, blocks, file_nums):
        df_temp = pd.DataFrame([{
            "Sub#": sub,
            "Session": session,
            "StimType": stim_type,
            "Amplitudes": amp,
            "Block": block,
            "File Num": file_num
        }])
        results_df = pd.concat([results_df, df_temp], ignore_index=True)

def get_stim_amplitude(sub, session, stim_type):
    mask = (
        (results_df["Sub#"] == sub) &
        (results_df["Session"] == session) &
        (results_df["StimType"] == stim_type)
    )
    matching_rows = results_df[mask]
    if not matching_rows.empty:
        return matching_rows["Amplitudes"].values[0]
    return None

triggers_df["Amplitude"] = triggers_df.apply(
    lambda row: get_stim_amplitude(row["Sub#"], row["Session"], row["StimType"]),
    axis=1
)

# Drop amplitudes from results_df before merge, like legacy code
results_df.drop("Amplitudes", axis=1, inplace=True, errors="ignore")

merged_stim_df = pd.merge(
    results_df,
    triggers_df,
    on=["Sub#", "Session", "StimType"],
    how="inner"
)

# Order columns to match your expected output
merged_stim_df = merged_stim_df[
    ["Sub#", "Session", "StimType", "Block", "File Num", "Time", "EventDescription", "Amplitude"]
].copy()

# Convert seconds to milliseconds
merged_stim_df["Time"] = merged_stim_df["Time"] * 1000.0

# -------------------------------------------------------
# EEG TIME COLUMN
# -------------------------------------------------------
sampling_rate = 1000
num_samples = len(eeg_df)

# At 1000 Hz, sample index = time in ms
eeg_df = eeg_df.copy()
eeg_df["Time"] = np.arange(num_samples, dtype=float)

# -------------------------------------------------------
# LEGACY LABEL EXTRACTION
# -------------------------------------------------------
data_as_list = [arr.tolist()[0] for arr in DSamp[0][0][5]]

eeg_label_df = pd.DataFrame(data_as_list, columns=["EEG Electrode Labels"])

unwanted_channels = ["BIP1", "BIP2", "RESP1"]
eeg_label_df = eeg_label_df[~eeg_label_df["EEG Electrode Labels"].isin(unwanted_channels)]

# -------------------------------------------------------
# CHECKS
# -------------------------------------------------------
print(merged_stim_df.head())
print(merged_stim_df.tail())
print(eeg_df.head())
print(eeg_df.tail())

/tmp/ipykernel_26226/1268117123.py:98: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results_df = pd.concat([results_df, df_temp], ignore_index=True)


  Sub# Session StimType Block File Num       Time EventDescription  Amplitude
0    1       1      M30     1      101   619499.0       Stim Start        1.0
1    1       1      M30     1      101   770515.0       Stim Start        1.0
2    1       1      M30     1      101   921515.0       Stim Start        1.0
3    1       1      M30     1      101  1072551.0       Stim Start        1.0
4    1       1      F30     2      101  1819593.0       Stim Start        1.0
  Sub# Session StimType Block File Num       Time EventDescription  Amplitude
3    1       1      M30     1      101  1072551.0       Stim Start        1.0
4    1       1      F30     2      101  1819593.0       Stim Start        1.0
5    1       1      F30     2      101  1970669.0       Stim Start        1.0
6    1       1      F30     2      101  2121644.0       Stim Start        1.0
7    1       1      F30     2      101  2272756.0       Stim Start        1.0
            Fp1           Fpz           Fp2           F7        

In [7]:
print(merged_stim_df.head())
print(merged_stim_df.tail())
print(eeg_df.head())
print(eeg_df.tail())

  Sub# Session StimType Block File Num       Time EventDescription  Amplitude
0    1       1      M30     1      101   619499.0       Stim Start        1.0
1    1       1      M30     1      101   770515.0       Stim Start        1.0
2    1       1      M30     1      101   921515.0       Stim Start        1.0
3    1       1      M30     1      101  1072551.0       Stim Start        1.0
4    1       1      F30     2      101  1819593.0       Stim Start        1.0
  Sub# Session StimType Block File Num       Time EventDescription  Amplitude
3    1       1      M30     1      101  1072551.0       Stim Start        1.0
4    1       1      F30     2      101  1819593.0       Stim Start        1.0
5    1       1      F30     2      101  1970669.0       Stim Start        1.0
6    1       1      F30     2      101  2121644.0       Stim Start        1.0
7    1       1      F30     2      101  2272756.0       Stim Start        1.0
            Fp1           Fpz           Fp2           F7        

# Make the eeg df a npy

In [15]:
# Extract EEG channel names (excluding 'Time')
eeg_channel_names = eeg_df.columns[:-1]

# Create a dictionary to store EEG data for each channel
eeg_data_dict = {}

# Populate the dictionary with EEG data
for channel in eeg_channel_names:
    eeg_data_dict[channel] = eeg_df[channel].values

# Convert the dictionary values to a numpy array
eeg_data_array = np.array([eeg_data_dict[channel] for channel in eeg_channel_names]).T

# Save the numpy array with EEG data as a single .npy file
save_path = '/home/a/projects/Complete-Neural-Signal-Analysis/eeg_data_with_channels.npy'
np.save(save_path, eeg_data_array)

In [18]:
# Specify the directory paths where you want to save the CSV files
merged_stim_directory = '/home/a/projects/Complete-Neural-Signal-Analysis/DataFrames'
eeg_directory = '/home/a/projects/Complete-Neural-Signal-Analysis/DataFrames'

# Save 'merged_stim_df' to CSV
merged_stim_df.to_csv(f"{merged_stim_directory}/merged_stim_df.csv", index=False)

# Save 'eeg_df' to CSV
eeg_df.to_csv(f"{eeg_directory}/eeg_df.csv", index=False)

print("DataFrames saved to CSV files successfully.")

DataFrames saved to CSV files successfully.


In [19]:
triggers = DSamp[0][0][0]
print("Triggers: ", triggers)

EEGdata = DSamp[0][0][1]
print("EEGdata: ", EEGdata)

fs = DSamp[0][0][2][0][0] 
print("fs: ", fs)

fsOld = DSamp[0][0][3][0][0] 
print("fsOld: ", fsOld)

time = DSamp[0][0][4][0]
print("Time: ", time)

label = DSamp[0][0][5]
print("Label: ", label)

nchan = DSamp[0][0][6][0][0]
print("nchan: ", nchan)

rate = DSamp[0][0][7][0][0]
print("Rate: ", rate)

npt = DSamp[0][0][8][0][0]
print("npt: ", npt)

Subj = DSamp[0][0][9][0]
print("Subj: ", Subj)

ptrackerPerf = DSamp[0][0][10]
print("PtrackerPerf: ", ptrackerPerf)

ptrackerTime = DSamp[0][0][11]
print("PtrackerTime: ", ptrackerTime)

ptrackerfs = DSamp[0][0][12][0][0]
print("Ptrackerfs: ", ptrackerfs)

Triggers:  [[[[20.428]
   [nan]
   [nan]
   ['Unknown']
   ['Unknown']
   ['Unknown']]

  [[619.442]
   [nan]
   [nan]
   ['Unknown']
   ['Unknown']
   ['Unknown']]

  [[619.499]
   [nan]
   [nan]
   ['Unknown']
   ['Unknown']
   ['M30']]

  [[654.746]
   [nan]
   [nan]
   ['Unknown']
   ['Unknown']
   ['Unknown']]

  [[770.515]
   [nan]
   [nan]
   ['Unknown']
   ['Unknown']
   ['M30']]

  [[805.571]
   [nan]
   [nan]
   ['Unknown']
   ['Unknown']
   ['Unknown']]

  [[921.515]
   [nan]
   [nan]
   ['Unknown']
   ['Unknown']
   ['M30']]

  [[956.651]
   [nan]
   [nan]
   ['Unknown']
   ['Unknown']
   ['Unknown']]

  [[1072.551]
   [nan]
   [nan]
   ['Unknown']
   ['Unknown']
   ['M30']]

  [[1107.578]
   [nan]
   [nan]
   ['Unknown']
   ['Unknown']
   ['Unknown']]

  [[1218.442]
   [nan]
   [nan]
   ['Unknown']
   ['Unknown']
   ['Unknown']]

  [[1817.46]
   [nan]
   [nan]
   ['Unknown']
   ['Unknown']
   ['Unknown']]

  [[1819.593]
   [nan]
   [nan]
   ['Unknown']
   ['Unknown']
   ['

In [ ]:
triggers = DSamp[0][0][0]
print("Triggers: ", triggers)

EEGdata = DSamp[0][0][1]
print("EEGdata: ", EEGdata)

fs = DSamp[0][0][2][0][0] 
print("fs: ", fs)

fsOld = DSamp[0][0][3][0][0] 
print("fsOld: ", fsOld)

time = DSamp[0][0][4][0]
print("Time: ", time)

label = DSamp[0][0][5]
print("Label: ", label)

nchan = DSamp[0][0][6][0][0]
print("nchan: ", nchan)

rate = DSamp[0][0][7][0][0]
print("Rate: ", rate)

npt = DSamp[0][0][8][0][0]
print("npt: ", npt)

Subj = DSamp[0][0][9][0]
print("Subj: ", Subj)

ptrackerPerf = DSamp[0][0][10]
print("PtrackerPerf: ", ptrackerPerf)

ptrackerTime = DSamp[0][0][11]
print("PtrackerTime: ", ptrackerTime)

ptrackerfs = DSamp[0][0][12][0][0]
print("Ptrackerfs: ", ptrackerfs)

Triggers:  [[(array([[20.428]]), array([[20429]], dtype=uint16), array(['0002'], dtype='<U4'), array([[2]], dtype=uint8), array(['Block Start'], dtype='<U11'), array([], shape=(1, 0), dtype=float64))
  (array([[619.442]]), array([[619443]], dtype=int32), array(['0002'], dtype='<U4'), array([[2]], dtype=uint8), array(['Block Start'], dtype='<U11'), array([], shape=(1, 0), dtype=float64))
  (array([[619.499]]), array([[619500]], dtype=int32), array(['0016'], dtype='<U4'), array([[16]], dtype=uint8), array(['Stim Start'], dtype='<U10'), array(['M30'], dtype='<U3'))
  (array([[654.746]]), array([[654747]], dtype=int32), array(['0032'], dtype='<U4'), array([[32]], dtype=uint8), array(['Stim Stop'], dtype='<U9'), array([], shape=(1, 0), dtype=float64))
  (array([[770.515]]), array([[770516]], dtype=int32), array(['0016'], dtype='<U4'), array([[16]], dtype=uint8), array(['Stim Start'], dtype='<U10'), array(['M30'], dtype='<U3'))
  (array([[805.571]]), array([[805572]], dtype=int32), array(['0

# load the csv's for the RNN from the Multifractal Analysis, and the eeg + stim csv's, after Multifractal Analysis

In [3]:
# Define the file paths
base_dir = 
eeg_df_path = base_dir + 'DataFrames/eeg_df.csv'
merged_stim_df_path = base_dir + 'DataFrames/merged_stim_df.csv'
hurst_exponents_path = base_dir + 'HurstExponents/hurst_exponents_df.csv'
rnn_mfdfa_X_path = base_dir + 'RNN_data/rnn_X_data_combined.npy'

# load data
eeg_df = pd.read_csv(eeg_df_path)
merged_stim_df = pd.read_csv(merged_stim_df_path)
hurst_exponents_df = pd.read_csv(hurst_exponents_path)
rnn_X_data_combined = np.load(rnn_mfdfa_X_path)

In [4]:
# Print the columns of eeg_df
print("Columns of eeg_df:")
print(eeg_df.columns)

# Print the columns of merged_stim_df
print("Columns of merged_stim_df:")
print(merged_stim_df.columns)

# Print the columns of hurst_exponents_df
print("Columns of hurst_exponents_df:")
print(hurst_exponents_df.columns)

# Assuming rnn_mfdfa_X_df is a NumPy array
print("Number of columns in rnn_X_data_combined:", rnn_X_data_combined.shape[1])

Columns of eeg_df:
Index(['Fp1', 'Fpz', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'FC5', 'FC1', 'FC2',
       'FC6', 'M1', 'T7', 'C3', 'Cz', 'C4', 'T8', 'M2', 'CP5', 'CP1', 'CP2',
       'CP6', 'P7', 'P3', 'Pz', 'P4', 'P8', 'POz', 'O1', 'Oz', 'O2', 'Time'],
      dtype='object')
Columns of merged_stim_df:
Index(['Sub#', 'Session', 'StimType', 'Block', 'File Num', 'Time',
       'EventDescription', 'Amplitude'],
      dtype='object')
Columns of hurst_exponents_df:
Index(['0'], dtype='object')
Number of columns in rnn_X_data_combined: 100


# Change everything to numerical

In [5]:
# Define mappings for frequency and location
frequency_mapping = {
    "F0": 0,
    "F5": 5,
    "F30": 30,
    "M0": 0,
    "M5": 5,
    "M30": 30,
    "P0": 0,
    "P5": 5,
    "P30": 30
}

location_mapping = {
    "F0": 1,
    "F5": 1,
    "F30": 1,
    "M0": 2,
    "M5": 2,
    "M30": 2,
    "P0": 3,
    "P5": 3,
    "P30": 3
}

# Check if 'StimType' column is present in the dataframe
if 'StimType' in merged_stim_df.columns:
    # Proceed with replacement of values and dropping the column
    merged_stim_df["Frequency"] = merged_stim_df["StimType"].replace(frequency_mapping)
    merged_stim_df["Location"] = merged_stim_df["StimType"].replace(location_mapping)
    merged_stim_df.drop('StimType', axis=1, inplace=True)
else:
    print("The 'StimType' column does not exist in the dataframe.")

# Replace "Stim Start" with 1 and "Stim Stop" with 2
merged_stim_df["EventDescription"] = merged_stim_df["EventDescription"].replace({
    "Stim Start": 1,
    "Stim Stop": 0
})

In [6]:
print(eeg_df['Time'].head())
print(merged_stim_df['Time'].head())

0    0.0
1    1.0
2    2.0
3    3.0
4    4.0
Name: Time, dtype: float64
0    619499.0
1    654746.0
2    770515.0
3    805571.0
4    921515.0
Name: Time, dtype: float64


In [7]:
# First, let's perform the merge operation
merged_eeg_stim_df = pd.merge_asof(eeg_df, merged_stim_df, on='Time', direction='backward')

# Create 'Stim' column based on 'EventDescription'.
# If 'EventDescription' is 1 (Stim start) and the 'Time' is >= 619499, we set 'Stim' as 1. Otherwise, 'Stim' is 0.
merged_eeg_stim_df['Stim'] = np.where((merged_eeg_stim_df['EventDescription'] == 1) & (merged_eeg_stim_df['Time'] >= 619499), 1, 0)

# Drop the 'EventDescription' column now.
merged_eeg_stim_df.drop(columns=['EventDescription'], inplace=True)

# Create a 'StimChange' column that's 1 where 'Stim' changes from 0 to 1, and 0 elsewhere
merged_eeg_stim_df['StimChange'] = (merged_eeg_stim_df['Stim'].diff() == 1).astype(int)

# Create a new 'block' column, incrementing by 1 each time 'StimChange' is 1 (i.e., each time a new stimulation session starts)
merged_eeg_stim_df['block'] = merged_eeg_stim_df['StimChange'].cumsum()

# Now we no longer need the 'StimChange' column, so we can drop it
merged_eeg_stim_df = merged_eeg_stim_df.drop('StimChange', axis=1)

# Reset the values for 'Amplitude', 'Frequency', 'Location', and 'block' when 'Stim' is 0
merged_eeg_stim_df.loc[merged_eeg_stim_df['Stim'] == 0, ['Amplitude', 'Frequency', 'Location', 'block']] = 0

# Now 'block' should be a new column in your DataFrame indicating the stimulation session (or "block") each row belongs to
print(merged_eeg_stim_df)


                  Fp1           Fpz           Fp2           F7          F3  \
0       -21295.988649 -20109.716727 -24153.383752  3189.340060  -45.189275   
1       -21303.747077 -20120.746154 -24163.864012  3178.880909  -56.702035   
2       -21315.466571 -20130.126577 -24171.944343  3164.903807  -69.465350   
3       -21317.809594 -20131.044726 -24174.790986  3159.478572  -73.214591   
4       -21325.798142 -20137.522181 -24179.985166  3144.934679  -84.871628   
...               ...           ...           ...          ...         ...   
4227783 -17297.981962 -16643.265483 -14783.694243   213.760887  123.981995   
4227784 -17288.547222 -16625.369538 -14763.506832   227.525891  141.239551   
4227785 -17286.892304 -16618.137233 -14755.598154   236.124689  150.365834   
4227786 -17281.277994 -16602.343699 -14743.081425   248.784515  164.864721   
4227787 -17330.522691 -16641.896722 -14783.662018   206.940363  122.044455   

                  Fz          F4           F8          FC5     

In [8]:
# Drop the old 'Block' column
merged_eeg_stim_df.drop(columns=['Block'], inplace=True)

# Dropping the 'File Num' column
merged_eeg_stim_df = merged_eeg_stim_df.drop('File Num', axis=1)

# Fill NaNs to 0's
merged_eeg_stim_df['Amplitude'] = merged_eeg_stim_df['Amplitude'].fillna(0)
merged_eeg_stim_df['Frequency'] = merged_eeg_stim_df['Frequency'].fillna(0)
merged_eeg_stim_df['Location'] = merged_eeg_stim_df['Location'].fillna(0)
merged_eeg_stim_df['block'] = merged_eeg_stim_df['block'].fillna(0)

# Changing all 'Sub#' values to 1
merged_eeg_stim_df['Sub#'] = 1

# Changing all 'Session' values to 1
merged_eeg_stim_df['Session'] = 1

# Show the resulting DataFrame
print(merged_eeg_stim_df)

                  Fp1           Fpz           Fp2           F7          F3  \
0       -21295.988649 -20109.716727 -24153.383752  3189.340060  -45.189275   
1       -21303.747077 -20120.746154 -24163.864012  3178.880909  -56.702035   
2       -21315.466571 -20130.126577 -24171.944343  3164.903807  -69.465350   
3       -21317.809594 -20131.044726 -24174.790986  3159.478572  -73.214591   
4       -21325.798142 -20137.522181 -24179.985166  3144.934679  -84.871628   
...               ...           ...           ...          ...         ...   
4227783 -17297.981962 -16643.265483 -14783.694243   213.760887  123.981995   
4227784 -17288.547222 -16625.369538 -14763.506832   227.525891  141.239551   
4227785 -17286.892304 -16618.137233 -14755.598154   236.124689  150.365834   
4227786 -17281.277994 -16602.343699 -14743.081425   248.784515  164.864721   
4227787 -17330.522691 -16641.896722 -14783.662018   206.940363  122.044455   

                  Fz          F4           F8          FC5     

In [12]:
# Assuming merged_eeg_stim_df is the DataFrame you are working with
# Assuming 'block' column represents the blocks

# Filter out block 0 (the non-stimulation block)
valid_blocks_df = merged_eeg_stim_df[merged_eeg_stim_df['block'] != 0]

# Find the number of unique valid blocks
num_valid_blocks = valid_blocks_df['block'].nunique()

# Print the number of unique valid blocks
print("Number of unique valid blocks of stimulation:", num_valid_blocks)


Number of unique valid blocks of stimulation: 9


In [ ]:
# Assuming merged_eeg_stim_df is the DataFrame you are working with
# Assuming 'Stim' column contains 1 for stimulation and 0 for no stimulation

# Define the number of rows you want to display before and after each stimulation block starts
num_rows_before_stim = 5
num_rows_after_stim = 5

# Function to extract rows before and after the stimulation block starts
def extract_rows_around_stim(merged_eeg_stim_df, stim_start_indices, num_rows_before_stim, num_rows_after_stim):
    for stim_start_idx in stim_start_indices:
        rows_before_stim = merged_eeg_stim_df.iloc[stim_start_idx - num_rows_before_stim:stim_start_idx]
        rows_after_stim = merged_eeg_stim_df.iloc[stim_start_idx:stim_start_idx + num_rows_after_stim]

        print(f"\nStimulation Block Start Index: {stim_start_idx}")
        print("Rows before the stimulation block starts:")
        print(rows_before_stim)

        print("\nRows after the stimulation block starts:")
        print(rows_after_stim)

# Find the indices where the stimulation block starts
stim_start_indices = merged_eeg_stim_df.index[merged_eeg_stim_df['Stim'] == 1].tolist()

# Call the function to extract rows for all stimulation blocks
extract_rows_around_stim(merged_eeg_stim_df, stim_start_indices, num_rows_before_stim, num_rows_after_stim)


In [20]:
# Show the resulting DataFrame
print(merged_eeg_stim_df)

                  Fp1           Fpz           Fp2           F7          F3  \
0       -21295.988649 -20109.716727 -24153.383752  3189.340060  -45.189275   
1       -21303.747077 -20120.746154 -24163.864012  3178.880909  -56.702035   
2       -21315.466571 -20130.126577 -24171.944343  3164.903807  -69.465350   
3       -21317.809594 -20131.044726 -24174.790986  3159.478572  -73.214591   
4       -21325.798142 -20137.522181 -24179.985166  3144.934679  -84.871628   
...               ...           ...           ...          ...         ...   
4227783 -17297.981962 -16643.265483 -14783.694243   213.760887  123.981995   
4227784 -17288.547222 -16625.369538 -14763.506832   227.525891  141.239551   
4227785 -17286.892304 -16618.137233 -14755.598154   236.124689  150.365834   
4227786 -17281.277994 -16602.343699 -14743.081425   248.784515  164.864721   
4227787 -17330.522691 -16641.896722 -14783.662018   206.940363  122.044455   

                  Fz          F4           F8          FC5     

In [19]:
# Make sure your DataFrame is sorted by 'Time'
merged_eeg_stim_df = merged_eeg_stim_df.sort_values('Time')

# Get the unique blocks
blocks = merged_eeg_stim_df['block'].unique()

# We only want to look at the first 9 blocks, excluding the first block (block 0)
blocks = blocks[1:10]

for block in blocks:
    # Get the indices of the rows belonging to this block
    block_indices = merged_eeg_stim_df[merged_eeg_stim_df['block'] == block].index
    
    # Get the index of the first row of the block
    first_index = block_indices[0]
    
    # Get the indices of the three rows before and three rows after the start of the block
    indices = range(first_index - 3, first_index + 3)
    
    # Select and print these rows
    print(merged_eeg_stim_df.loc[indices])

                 Fp1           Fpz           Fp2           F7         F3  \
619496 -21514.565997 -19579.117859 -22808.936192  2773.475892 -35.557832   
619497 -21493.753131 -19558.399511 -22790.959384  2795.570464 -14.778337   
619498 -21495.608737 -19567.024258 -22800.700106  2790.819965 -20.460584   
619499 -21494.610241 -19570.985140 -22801.964889  2787.997341 -25.947773   
619500 -21489.067781 -19569.220022 -22800.405633  2786.277971 -20.967430   
619501 -21492.449675 -19575.965596 -22808.633861  2766.501341 -30.273663   

                 Fz          F4           F8          FC5          FC1  ...  \
619496 -8483.830863 -503.396550  3346.327936  6651.888843  7001.381451  ...   
619497 -8465.880712 -489.352664  3366.895072  6673.518314  7020.005337  ...   
619498 -8474.053096 -500.904302  3346.567180  6670.705008  7012.466795  ...   
619499 -8477.884632 -501.414926  3350.030884  6667.162657  7004.993011  ...   
619500 -8475.553025 -498.035359  3356.795540  6669.650655  7013.918986  

In [18]:
# Save the DataFrame as a CSV file
merged_eeg_stim_df.to_csv('', index=False)

In [ ]:
 Hjorth Coefficients,, Discrete Wavelet Transform, Differential Asymmetry, Magnitude Squared Coherence Estimate